# 01 — Data Exploration

Validate Alpaca pulls, inspect FRED series, check data quality.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils.seeds import set_all_seeds
set_all_seeds()
%matplotlib inline

## 1. Alpaca Smoke Test — SPY

In [ ]:
from src.data.alpaca_loader import smoke_test
spy = smoke_test('SPY', n_rows=10)
spy

## 2. Load Full Universe

In [ ]:
import yaml
with open('../config/assets.yaml') as f:
    assets = yaml.safe_load(f)

from src.data.alpaca_loader import load_ticker
spy_full = load_ticker('SPY', start='2016-01-01')
print(f'SPY: {spy_full.shape}, {spy_full.index[0].date()} to {spy_full.index[-1].date()}')
spy_full.tail()

## 3. FRED Validation

In [ ]:
# Run pre-flight validation -- will raise SystemExit if any series fails
from src.data.validate_fred import validate_all
results = validate_all(force_refresh=False)
ok = sum(results.values())
print(f'{ok}/{len(results)} series validated.')

## 4. Price Coverage Heatmap

In [ ]:
from src.data.alpaca_loader import load_universe
tickers = assets['sector_etfs'] + assets['indices'] + assets['commodities']
ohlcv = load_universe(tickers, start='2016-01-01')

coverage = pd.DataFrame({
    t: pd.Series({'start': df.index[0].date(), 'end': df.index[-1].date(), 'n': len(df)})
    for t, df in ohlcv.items()
}).T
coverage

## 5. Return Distribution by Sector

In [ ]:
import numpy as np
sector_rets = pd.DataFrame({
    t: np.log(ohlcv[t]['close'] / ohlcv[t]['close'].shift(1))
    for t in assets['sector_etfs'] if t in ohlcv
})

fig, ax = plt.subplots(figsize=(14, 4))
sector_rets.plot.box(ax=ax, showfliers=False)
ax.set_title('Daily Return Distribution by Sector ETF')
ax.set_ylabel('Daily Log Return')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()